# Sub-1B document-VLM comparison on a T4 GPU

Thin runner: **clone → install → run repo scripts**. Runs every sub-1B model + **PaddleOCR-VL 1.0/1.5/1.6** on the capability, spatial/context, and **proposed custom-eval** sets, measuring **score + inference time + CPU/GPU memory**.

Runtime → Change runtime type → **T4 GPU**, then Run all.


## 1. GPU check

In [ ]:
!nvidia-smi -L

## 2. Clone repo + install

In [ ]:
%cd /content
![ -d OCR ] || git clone https://github.com/SangbumChoi/OCR.git
%cd /content/OCR
!git checkout claude/new-session-w79q0i && git pull --ff-only
!pip -q install -e '.[models,finetune]' protobuf

## 3. Run the full comparison (chat VLMs @ tf4.49, PaddleOCR-VL @ tf4.57; all measured)
Installs CJK fonts + QR/barcode libs, builds the probes incl. the custom-eval set, runs all models on capability + spatial/context + custom-eval, and aggregates.

In [ ]:
!DEVICE=cuda bash scripts/run_full_comparison.sh

## 4. Scores + efficiency (time & memory)

In [ ]:
print(open('docs/results/matrix_capability.md').read())

## 5. Spatial / context shortcut-robust signals

In [ ]:
print(open('docs/results/matrix_probe.md').read())
!python scripts/analyze_probe_signals.py --probe probe

## 6. Proposed custom-eval — by class / language / rotation / direction / spotting

In [ ]:
print(open('docs/results/custom_eval_breakdown.md').read())

## 7. PaddleOCR-VL 1.0 vs 1.5 vs 1.6

In [ ]:
!for m in paddleocr-vl paddleocr-vl-1.5 paddleocr-vl-1.6; do echo "== $m =="; python3 -c "import json;s=json.load(open(f'docs/results/$m/custom_eval/summary.json'));print({k:s.get(k) for k in ['score','avg_latency_s','peak_gpu_mb']})" 2>/dev/null || echo 'n/a'; done

## 8. Download all results

In [ ]:
!zip -qr /content/docvlm_results.zip results
from google.colab import files; files.download('/content/docvlm_results.zip')